# MockMate Gemma — Privacy-First Interviewer**MockMate Gemma is a mock interview coach that runs entirely on your own machine.**You speak an answer out loud; a local `llama-server` running **Gemma 4 E4B** transcribes it,and the *same* model grades it against a real rubric. The audio never leaves the machine, andit is not thrown away before judging either — the closing seconds of the answer are handed tothe model as audio alongside the transcript, so the grade is formed from what was actuallysaid rather than from a lossy text summary of it.> ### ⚠️ What this notebook does and does not touch>> **This notebook processes only the fixture audio shipped with it. It has no microphone> input. Internet is disabled. Nothing in this notebook uploads audio, transcripts, or> scores anywhere.**>> The microphone path exists only in the local install (`sidecar/`), which binds `127.0.0.1`> and is not started here. Every WAV read below comes from the attached `mockmate-fixtures`> dataset — recordings the team made of themselves, shipped deliberately so a judge can> evaluate the project without hardware and without ever handing us their voice.**Track:** Privacy-First Interviewer · [Build with Gemma — ML Nashik](https://www.kaggle.com/competitions/build-with-gemma-ml-nashik/)**Repo:** https://github.com/varadshajith/mockmate-gemma**Team:** Varad — architecture & inference · Akshada — interface & UI testing ·Tanvi — interview content & rubrics · Mayur — model testing & optimization---### How to read the numbers in this notebookEvery number printed below was produced by the cell that printed it, in this run, on thismachine. Nothing is hardcoded from an earlier session. Where something could not run, thecell says so and prints no number at all — see the run checklist in the final cell forexactly which parts executed live.

## 1 · Environment probeWhat hardware are we actually on? The compute capability is read from the GPU, not assumed — the writeup's latency figures come from an RTX 4050 (compute 89) and a Kaggle T4 is compute 75. Building for the wrong architecture either fails or silently falls back.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time# One place records what actually ran, so the final cell can print an honest# checklist instead of us claiming from memory. Every entry is written by the# cell that did (or did not do) the work.RUN_LOG = {}def log(step, status, detail=""):    RUN_LOG[step] = {"status": status, "detail": detail}    return statusdef sh(cmd, **kw):    """Run a shell command, return (returncode, stdout+stderr)."""    p = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)    return p.returncode, (p.stdout or "") + (p.stderr or "")rc, smi = sh("nvidia-smi")print(smi if rc == 0 else "nvidia-smi not available — no NVIDIA GPU visible to this container.")# Compute capability straight from the driver. Do NOT hardcode this.CUDA_ARCH = NoneGPU_NAME = Nonerc, out = sh("nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader")if rc == 0 and out.strip():    GPU_NAME, cap, vram = [f.strip() for f in out.strip().splitlines()[0].split(",")]    CUDA_ARCH = cap.replace(".", "")    print(f"GPU            : {GPU_NAME}")    print(f"compute_cap    : {cap}  ->  -DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}")    print(f"VRAM           : {vram}")else:    print("Could not read compute capability — the build cell will fall back to CPU-only.")rc, nvcc = sh("nvcc --version")print(f"\nnvcc           : {nvcc.strip().splitlines()[-1] if rc == 0 else 'not found'}")print(f"python         : {sys.version.split()[0]}")print(f"platform       : {platform.platform()}")print(f"cpu cores      : {os.cpu_count()}")NODE_BIN = shutil.which("node")rc, node_v = sh([NODE_BIN, "--version"]) if NODE_BIN else (1, "")print(f"node           : {node_v.strip() if rc == 0 else 'NOT FOUND — cells 8/9/11 cannot run'}")log("env_probe", "ran", f"gpu={GPU_NAME} arch={CUDA_ARCH} node={bool(NODE_BIN)}")

## 2 · Verify the attached inputsFive datasets have to be attached for this notebook to run end to end. If one is missing, this cell names it — a judge who forgot to attach something should be told which one, not handed a stack trace 12 minutes into a build.The repo source is fatal: nothing downstream works without it, so that raises immediately. A missing *model* is not fatal — the notebook degrades down the fallback ladder documented in `notebooks/README.md` and every skipped cell prints that it was skipped.

In [ ]:
import glob, pathlib, zipfileINPUT = pathlib.Path("/kaggle/input")WORK = pathlib.Path("/kaggle/working")def find_one(pattern):    """First path matching a recursive glob under /kaggle/input, or None."""    hits = sorted(glob.glob(str(INPUT / "**" / pattern), recursive=True))    return pathlib.Path(hits[0]) if hits else None# Located by filename rather than by dataset directory: Kaggle Models and Kaggle# Datasets mount at different paths, and the Gemma GGUF may arrive as either.EXPECTED = {    "mockmate-gemma-src   (repo zip)":        ("*mockmate*.zip",                      True),    "mockmate-fixtures    (16k mono WAVs)":   ("*.wav",                               False),    "llamacpp-src         (pinned source)":   ("llama.cpp*",                          True),    "gemma 4 E4B          (weights)":         ("gemma-4-E4B-it-Q4_0.gguf",            False),    "gemma 4 E4B          (mmproj)":          ("mmproj-gemma-4-E4B-it-Q8_0.gguf",     False),    "bge-small-en-v1.5    (embeddings)":      ("bge-small-en-v1.5-q8_0.gguf",         False),}found = {}print(f"{'input':<38} {'status':<9} path")print("-" * 100)for label, (pattern, fatal) in EXPECTED.items():    hit = find_one(pattern)    found[label] = hit    print(f"{label:<38} {'FOUND' if hit else 'MISSING':<9} {hit if hit else '—'}")missing_fatal = [l for l, (p, f) in EXPECTED.items() if f and not found[l]]missing_soft = [l for l, (p, f) in EXPECTED.items() if not f and not found[l]]if missing_fatal:    raise RuntimeError(        "Cannot continue. Attach these datasets via the notebook sidebar "        "(+ Add Input), then re-run:\n  - " + "\n  - ".join(missing_fatal) +        "\nSlugs and attach instructions are in notebooks/README.md."    )if missing_soft:    print("\n" + "!" * 100)    print("MISSING (not fatal — the notebook will degrade and say so):")    for m in missing_soft:        print(f"  - {m}")    print("Attach these via + Add Input for the full demo. See notebooks/README.md.")    print("!" * 100)# Unpack the repo at the demo commit into the writable working dir.REPO = WORK / "mockmate"if not REPO.exists():    with zipfile.ZipFile(found["mockmate-gemma-src   (repo zip)"]) as z:        z.extractall(WORK / "_src")    roots = [p for p in (WORK / "_src").rglob("AGENTS.md")]    if not roots:        raise RuntimeError("mockmate-gemma-src zip does not contain AGENTS.md — wrong archive?")    roots[0].parent.rename(REPO)print(f"\nrepo unpacked  : {REPO}")GEMMA_GGUF = found["gemma 4 E4B          (weights)"]MMPROJ_GGUF = found["gemma 4 E4B          (mmproj)"]BGE_GGUF = found["bge-small-en-v1.5    (embeddings)"]FIXTURES = sorted(glob.glob(str(INPUT / "**" / "*.wav"), recursive=True))print(f"fixture WAVs   : {len(FIXTURES)}")for f in FIXTURES:    print(f"                 {pathlib.Path(f).name}")HAVE_MODELS = bool(GEMMA_GGUF and MMPROJ_GGUF)HAVE_EMBED = bool(BGE_GGUF)log("verify_inputs", "ran", f"models={HAVE_MODELS} embed={HAVE_EMBED} fixtures={len(FIXTURES)}")

## 3 · Build llama.cpp from the pinned sourceBuilt here rather than pip-installed, because the audio path this project depends on (`input_audio` on `/v1/chat/completions`) and the top-level `grammar` field are both build-specific. Expect **8–15 minutes**.If the CUDA build fails, the cell falls back to a CPU-only build and *labels every latency number below as CPU-only*. A CPU number compared against the GPU numbers in the writeup would be a fabricated comparison.

In [ ]:
import shutil, time, pathlib, tarfile, zipfileLLAMA_SRC = WORK / "llama.cpp"BUILD_DIR = WORK / "llama.cpp-build"BUILD_MODE = None      # "cuda" | "cpu" | NoneBUILD_SECONDS = Nonesrc_hit = found["llamacpp-src         (pinned source)"]if not LLAMA_SRC.exists():    if src_hit.is_dir():        shutil.copytree(src_hit, LLAMA_SRC)    elif src_hit.suffix == ".zip":        with zipfile.ZipFile(src_hit) as z:            z.extractall(WORK / "_llama")        LLAMA_SRC = next(p.parent for p in (WORK / "_llama").rglob("CMakeLists.txt"))    else:        with tarfile.open(src_hit) as t:            t.extractall(WORK / "_llama")        LLAMA_SRC = next(p.parent for p in (WORK / "_llama").rglob("CMakeLists.txt"))print(f"llama.cpp source: {LLAMA_SRC}")# LLAMA_CURL=OFF is required: the default build links libcurl for model# downloading, which is exactly the capability this notebook must not have.COMMON = [    f"-S {LLAMA_SRC}", f"-B {BUILD_DIR}",    "-DCMAKE_BUILD_TYPE=Release",    "-DLLAMA_CURL=OFF",    "-DLLAMA_BUILD_TESTS=OFF",]def try_build(flags, label):    global BUILD_SECONDS    shutil.rmtree(BUILD_DIR, ignore_errors=True)    started = time.time()    rc, out = sh("cmake " + " ".join(COMMON + flags))    if rc != 0:        print(out[-3000:])        return False, time.time() - started    rc, out = sh(f"cmake --build {BUILD_DIR} --config Release -j {os.cpu_count()} --target llama-server")    elapsed = time.time() - started    if rc != 0:        print(out[-3000:])        return False, elapsed    return True, elapsedif CUDA_ARCH:    print(f"Building with CUDA for compute {CUDA_ARCH} …")    ok, BUILD_SECONDS = try_build([f"-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}"], "cuda")    if ok:        BUILD_MODE = "cuda"else:    ok = Falseif not ok:    print("\n" + "!" * 100)    print("CUDA BUILD DID NOT SUCCEED — falling back to a CPU-only build.")    print("Every latency number printed later in this notebook will therefore be CPU-only")    print("and is NOT comparable to the GPU numbers in the project writeup.")    print("!" * 100 + "\n")    ok, BUILD_SECONDS = try_build(["-DGGML_CUDA=OFF"], "cpu")    BUILD_MODE = "cpu" if ok else NoneSERVER_BIN = BUILD_DIR / "bin" / "llama-server"if not (ok and SERVER_BIN.exists()):    raise RuntimeError("llama-server did not build — see the cmake output above.")print(f"build mode     : {BUILD_MODE.upper()}")print(f"build duration : {BUILD_SECONDS/60:.1f} min ({BUILD_SECONDS:.0f}s)")print(f"binary         : {SERVER_BIN}")LATENCY_CAVEAT = "" if BUILD_MODE == "cuda" else "  [CPU-ONLY BUILD — not comparable to GPU numbers]"log("build_llamacpp", "ran", f"mode={BUILD_MODE} seconds={BUILD_SECONDS:.0f}")

## 4 · Launch both local serversTwo servers, exactly as the local install runs them (see the Quickstart in the repo README): Gemma on `127.0.0.1:8080` for transcription and grading, and a small embedding model on `127.0.0.1:8081` for memory retrieval. Both bind loopback. Neither is reachable from outside this container.

In [ ]:
import socket, urllib.request, urllib.errorGEMMA_URL = "http://127.0.0.1:8080"EMBED_URL = "http://127.0.0.1:8081"LOGS = WORK / "logs"LOGS.mkdir(exist_ok=True)procs = {}def wait_health(base, timeout_s, label):    """Poll /health until the server reports ready. Returns seconds, or None."""    started = time.time()    while time.time() - started < timeout_s:        if label in procs and procs[label].poll() is not None:            return None                     # process died; caller reads the log        try:            with urllib.request.urlopen(f"{base}/health", timeout=5) as r:                if r.status == 200:                    return time.time() - started        except Exception:            time.sleep(2)    return Nonedef launch(label, args, base, timeout_s):    logfile = open(LOGS / f"{label}.log", "w")    procs[label] = subprocess.Popen([str(SERVER_BIN)] + args, stdout=logfile, stderr=subprocess.STDOUT)    ready = wait_health(base, timeout_s, label)    return readyGEMMA_READY = EMBED_READY = NoneFLASH_ATTN = Noneif HAVE_MODELS:    gemma_args = [        "-m", str(GEMMA_GGUF), "--mmproj", str(MMPROJ_GGUF),        "-ngl", "99", "-c", "16384",        "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",        "--host", "127.0.0.1", "--port", "8080",    ]    GEMMA_READY = launch("gemma", gemma_args + ["--flash-attn", "on"], GEMMA_URL, 900)    FLASH_ATTN = "on"    if GEMMA_READY is None:        # Older/newer builds and some GPUs reject the flash-attn flag outright.        # Retrying without it is a real difference in how the server was run,        # so it gets printed rather than silently absorbed.        print("Gemma did not come up with --flash-attn on. Tail of its log:")        print((LOGS / "gemma.log").read_text()[-1500:])        print("\nRetrying WITHOUT flash-attn …")        procs["gemma"].kill()        GEMMA_READY = launch("gemma", gemma_args, GEMMA_URL, 900)        FLASH_ATTN = "off (retried — the flag was rejected on this GPU)"    print(f"gemma  8080    : {'ready in %.1fs' % GEMMA_READY if GEMMA_READY else 'FAILED TO START'}{LATENCY_CAVEAT}")    print(f"flash-attn     : {FLASH_ATTN}")else:    print("gemma  8080    : SKIPPED — model GGUF not attached.")if HAVE_EMBED:    EMBED_READY = launch("embed", [        "-m", str(BGE_GGUF),        "--embedding",        # --pooling mean is MANDATORY. Without it llama-server returns one        # vector PER TOKEN instead of one per sentence, src/memory.js takes        # index [0], and every cosine similarity below silently becomes a        # comparison of two [CLS] tokens. The numbers still look plausible.        # This cost us a full debugging round. Do not remove it.        "--pooling", "mean",        "-ngl", "0",        "--host", "127.0.0.1", "--port", "8081",    ], EMBED_URL, 300)    print(f"embed  8081    : {'ready in %.1fs' % EMBED_READY if EMBED_READY else 'FAILED TO START'}")else:    print("embed  8081    : SKIPPED — bge GGUF not attached.")SERVERS_UP = GEMMA_READY is not NoneEMBED_UP = EMBED_READY is not Nonelog("launch_servers", "ran", f"gemma={GEMMA_READY} embed={EMBED_READY} flash_attn={FLASH_ATTN}")

## 5 · Who is allowed to call the modelThis is copied verbatim from [`AGENTS.md`](https://github.com/varadshajith/mockmate-gemma/blob/main/AGENTS.md).It is an exhaustive list — no other file in the project may call an inference server directly.| # | Caller | Server | Endpoints | Purpose ||---|---|---|---|---|| 1 | `src/llm.js` | llama-server `127.0.0.1:8080` | `/completion` (text-only, raw GBNF grammar)<br>`/v1/chat/completions` (multimodal, top-level `grammar` field) | Grading, follow-ups, recommendations, question generation, contradiction checking, slot checking. May *receive* base64 audio as an input to grading. May **not** capture, resample, or otherwise process audio. || 2 | `sidecar/` | llama-server `127.0.0.1:8080` | `/v1/chat/completions` | Transcription only. Never grades, never generates interview content. || 3 | `src/memory.js` | embedding server `127.0.0.1:8081` | `/embedding` | Retrieval only. This server never generates text. |**Why the rule exists:** two of this project's tests silently broke because they built theirown copy of a prompt instead of importing the real one, and spent two rounds of debuggingreporting drift between two hand-maintained copies as if it were a regression. A singleboundary per capability is what makes a test able to measure the thing it claims to measure.The same rule binds this notebook. The demo cells below **import** the real modules and shellout to them; they do not restate a prompt, a rubric, a grammar, a threshold, or a similarityformula. See `notebooks/kaggle_bridge.js` — it is deliberately just a pass-through.

## 6 · DEMO — transcription on fixture audioOne fixture WAV goes to `/v1/chat/completions` as `input_audio`. The request body is built by `sidecar/transcribe._build_payload()` — the same function the live app uses — so the instruction text and sampling parameters here are the real ones, not a notebook restatement of them.

In [ ]:
import base64, wave, urllib.requestTRANSCRIPT = NoneFIXTURE = Noneif not (SERVERS_UP and FIXTURES):    print("DID NOT RUN — " + ("no fixture WAVs attached." if SERVERS_UP else "Gemma server is not up."))    log("demo_transcribe", "skipped", "server or fixtures unavailable")else:    sys.path.insert(0, str(REPO / "sidecar"))    import transcribe as sidecar_transcribe   # the real caller from AGENTS.md row 2    FIXTURE = pathlib.Path(FIXTURES[0])    wav_bytes = FIXTURE.read_bytes()    with wave.open(str(FIXTURE)) as w:        audio_seconds = w.getnframes() / w.getframerate()        print(f"fixture        : {FIXTURE.name}")        print(f"format         : {w.getframerate()} Hz, {w.getnchannels()} ch, {audio_seconds:.1f}s")    # _build_payload carries two settings that are load-bearing, both documented    # at length in sidecar/transcribe.py:    #   temperature 0.2  — at 1.0, ~1 request in 5 stalls to ~10s instead of ~2.3s    #   chat_template_kwargs.enable_thinking = False — WITHOUT THIS the model    #   spends its token budget on a chain-of-thought block and returns a    #   TRUNCATED transcript while still reporting finish_reason "stop".    #   The response looks complete and is not. Do not remove it, and do not    #   substitute "reasoning_budget": 0 — that is silently ignored on this build.    payload = sidecar_transcribe._build_payload(wav_bytes)    print(f"temperature    : {payload['temperature']}")    print(f"enable_thinking: {payload['chat_template_kwargs']['enable_thinking']}")    req = urllib.request.Request(        f"{GEMMA_URL}/v1/chat/completions",        data=json.dumps(payload).encode(),        headers={"Content-Type": "application/json"},    )    started = time.time()    with urllib.request.urlopen(req, timeout=180) as r:        body = json.load(r)    latency = time.time() - started    choice = body["choices"][0]    TRANSCRIPT = choice["message"]["content"].strip()    completion_tokens = body.get("usage", {}).get("completion_tokens")    print(f"\nfinish_reason  : {choice['finish_reason']}")    print(f"latency        : {latency:.2f}s{LATENCY_CAVEAT}")    print(f"audio duration : {audio_seconds:.2f}s")    print(f"realtime factor: {audio_seconds/latency:.2f}x  (>1 means faster than realtime)")    if completion_tokens:        print(f"gen tokens     : {completion_tokens}")        print(f"tokens/sec     : {completion_tokens/latency:.1f}{LATENCY_CAVEAT}")    else:        print("gen tokens     : not reported by this build — tokens/sec not computed")    print("\n--- transcript -------------------------------------------------------------")    print(TRANSCRIPT)    print("----------------------------------------------------------------------------")    log("demo_transcribe", "ran", f"latency={latency:.2f}s chars={len(TRANSCRIPT)}")

## 7 · DEMO — hybrid grading (transcript **and** audio)This is the core Gemma-integration claim: the same model that transcribed the answer also grades it, and it grades from the audio as well as the text. Most pipelines transcribe, throw the audio away, and judge a string. Here the closing portion of the answer is attached as `input_audio` alongside the full transcript, so anything the transcript could not carry is still in front of the grader.The call goes through `src/llm.js` → `evaluate()`. The notebook does not build the prompt, the rubric, or the GBNF grammar — it hands `evaluate()` a request and prints what comes back.

In [ ]:
VERDICT = NoneGRADE_REQUEST = Nonedef run_bridge(command, payload, timeout=300):    """Call notebooks/kaggle_bridge.js. Returns the parsed JSON it printed."""    p = subprocess.run(        [NODE_BIN, str(REPO / "notebooks" / "kaggle_bridge.js"), command],        input=json.dumps(payload), capture_output=True, text=True, timeout=timeout, cwd=str(REPO),    )    try:        return json.loads(p.stdout)    except json.JSONDecodeError:        return {"ok": False, "error": f"bridge produced no JSON (exit {p.returncode}): {p.stderr[-800:]}"}if not (SERVERS_UP and TRANSCRIPT and NODE_BIN):    reason = ("node is not installed on this image" if not NODE_BIN              else "no transcript from the previous cell" if SERVERS_UP else "Gemma server is not up")    print(f"DID NOT RUN — {reason}.")    log("demo_grading", "skipped", reason)else:    questions = json.loads((REPO / "data" / "interview_questions.json").read_text())    # Which question this fixture is an answer to is recorded in    # notebooks/fixtures/MANIFEST.md and encoded in the fixture filename.    qid = FIXTURE.stem    pool = [q for d in questions["domains"].values() for q in d]    question = next((q for q in pool if q["id"] == qid), pool[0])    if question["id"] != qid:        print(f"NOTE: no question id matches fixture '{qid}'; grading against '{question['id']}' instead.")    # Fixtures are <=30s by construction (sidecar/config.MAX_CHUNK_SECONDS is an    # inviolable ceiling — past 30s the model silently truncates the audio and    # still reports finish_reason "stop"), so the whole clip IS the final <=30s    # and no trimming happens here. Audio processing belongs in sidecar/, never    # in a caller. See AGENTS.md.    GRADE_REQUEST = {        "question": question["question"],        "category": question["shape"],        "modelAnswer": question.get("reference_answers", {}).get("score_9", ""),        "userAnswer": TRANSCRIPT,        "probeUsed": False,        "audioB64": base64.b64encode(FIXTURE.read_bytes()).decode("ascii"),    }    print(f"question       : {question['id']} — {question['question']}")    print(f"category       : {question['shape']}")    print(f"transcript     : {len(TRANSCRIPT)} chars sent as text")    print(f"audio          : {FIXTURE.name} sent as input_audio ({audio_seconds:.1f}s)\n")    out = run_bridge("grade", {"request": GRADE_REQUEST})    if not out.get("ok"):        # Rule 1: on any failure we print the failure. We never invent a score.        print("GRADING FAILED — no score is being shown, deliberately.")        print(out.get("error"))        log("demo_grading", "failed", out.get("error", "")[:200])    else:        VERDICT = out["result"]        print(f"latency        : {out['latencyMs']/1000:.2f}s{LATENCY_CAVEAT}")        print(f"endpoint       : {out['endpoint']}")        print(f"temperature    : {out['temperatureSent']}  (read from src/llm.js, not set here)")        print(f"gradedFrom     : {VERDICT['gradedFrom']}   <-- 'audio+text' means the audio "              f"reached the grader")        print(f"score          : {VERDICT['score']}")        print(f"levelSignal    : {VERDICT['levelSignal']}")        print("\n--- parsed JSON verdict ----------------------------------------------------")        print(json.dumps(VERDICT, indent=2))        print("----------------------------------------------------------------------------")        if VERDICT["gradedFrom"] != "audio+text":            print("\nNOTE: gradedFrom is 'text', so the audio path errored and evaluate() fell "                  "back to text-only grading. The console warning explaining why is in the "                  "bridge's stderr. This is a real result, not a hidden one.")        log("demo_grading", "ran", f"score={VERDICT['score']} gradedFrom={VERDICT['gradedFrom']}")

## 8 · DEMO — determinism proof**The single most important cell in this notebook.** The identical grading call, six times: three at the temperature `src/llm.js` actually uses, then three at `0.2`. The prompt, the grammar and the endpoint are byte-identical across all six — the only thing that changes is one sampling number, rewritten on the way out by a `fetch` wrapper in `kaggle_bridge.js` so that no second copy of the prompt exists.

In [ ]:
def three_runs(label, temperature):    scores, signals = [], []    for i in range(3):        payload = {"request": GRADE_REQUEST}        if temperature is not None:            payload["temperature"] = temperature        out = run_bridge("grade", payload)        if not out.get("ok"):            print(f"  run {i+1}: FAILED — {out.get('error')}")            scores.append(None); signals.append(None)            continue        scores.append(out["result"]["score"])        signals.append(out["result"]["levelSignal"])        print(f"  run {i+1}: score={out['result']['score']:>3}  "              f"levelSignal={out['result']['levelSignal']:<10} "              f"temp_sent={out['temperatureSent']}  {out['latencyMs']/1000:.1f}s")    real = [s for s in scores if s is not None]    spread = (max(real) - min(real)) if len(real) > 1 else None    print(f"  -> scores {scores}   spread: {spread if spread is not None else 'n/a'}\n")    return scores, spreadif VERDICT is None:    print("DID NOT RUN — cell 7 produced no verdict, so there is nothing to re-run identically.")    log("demo_determinism", "skipped", "no verdict from grading cell")else:    print(f"Temperature as shipped in src/llm.js (evaluate() hardcodes it):")    det_scores, det_spread = three_runs("shipped", None)    print(f"Temperature forced to 0.2 — identical prompt, identical grammar:")    hot_scores, hot_spread = three_runs("0.2", 0.2)    print("=" * 78)    print(f"{'':<26}{'run 1':>8}{'run 2':>8}{'run 3':>8}{'spread':>10}")    print(f"{'temperature 0.0 (shipped)':<26}" + "".join(f"{s if s is not None else 'ERR':>8}" for s in det_scores) + f"{det_spread if det_spread is not None else 'n/a':>10}")    print(f"{'temperature 0.2':<26}" + "".join(f"{s if s is not None else 'ERR':>8}" for s in hot_scores) + f"{hot_spread if hot_spread is not None else 'n/a':>10}")    print("=" * 78)    log("demo_determinism", "ran", f"t0={det_scores} t02={hot_scores}")

### Why this cell is the one that mattersWe found our own grader swinging **ten points on identical input** — the same answer scored65, then 75, then 75 across three runs. Sampling temperature is a sensible default almosteverywhere in an LLM app; it is exactly wrong for a grader. There is no creative upside to arandomly sampled score, and the downside is a tool that lies about its own consistency.The project this one was forked from produced its scores with `Math.random()`. **A score thatchanges on re-run is a different flavour of that same bug, and the same category of harm** — acandidate is being told a number about their own performance that does not mean what theythink it means. Setting `temperature: 0.0` in `evaluate()` is not a tuning preference. It isthe difference between a measurement and a plausible-looking number.Whatever the table above prints is what happened in this run, on this GPU. If the spread at`0.0` is not zero, that is a real finding about this build and we would rather show it thanhide it.

## 9 · DEMO — the thresholds live in JavaScript, not in the modelThe model decides the score. What that score *means* — advance, hold, probe, step down — is decided by four lines of arithmetic in `src/llm.js`. The prompt does ask the model for a `levelSignal`, and `evaluate()` throws the model's answer away in favour of the derived one, because in testing the model would not reliably apply the *"a probe has already been used"* clause.The source printed below is sliced out of `src/llm.js` at run time and evaluated — the notebook is running the bytes on disk, not a retyped copy of them.

In [ ]:
if VERDICT is None or not NODE_BIN:    print("DID NOT RUN — " + ("node is not installed on this image." if not NODE_BIN                              else "no score from the grading cell to feed in."))    log("demo_thresholds", "skipped", "no score or no node")else:    print("--- src/llm.js, read from disk at run time ---------------------------------")    for probe_used in (False, True):        out = run_bridge("level", {"score": VERDICT["score"], "probeUsed": probe_used}, timeout=30)        if probe_used is False:            print(out["source"])            print("---------------------------------------------------------------------------\n")        print(f"deriveLevelSignal(score={out['score']}, probeUsed={str(probe_used):<5}) -> {out['levelSignal']!r}")    print(f"\nevaluate() returned levelSignal   : {VERDICT['levelSignal']!r}")    print("Same value, because evaluate() calls this function rather than trusting the model.")    log("demo_thresholds", "ran", f"score={VERDICT['score']}")

**The model judges. JavaScript decides what the judgement means** — because a model cannot reliably apply *"if the score is under 60 and a probe was already used, step down"*, and a rule that is applied unreliably is not a rule.

## 10 · DEMO — memory retrievalWhen an answer scores badly, the app stores a compact record of the weakness and embeds it. Later sessions retrieve it by meaning, not by keyword. Below: one query embedded against eight stored topics — six from the interview domain, two deliberately from nowhere near it.The threshold and the cosine function both come from `src/memory.js`. The notebook supplies only the strings.

In [ ]:
QUERY = "I couldn't explain the trade-offs when choosing a database index"TOPICS = [    {"label": "database indexing",      "text": "database indexing trade-offs and when an index hurts writes", "inDomain": True},    {"label": "sql vs nosql",           "text": "choosing between SQL and NoSQL for a changing data model",     "inDomain": True},    {"label": "query optimisation",     "text": "slow query optimisation and reading an execution plan",        "inDomain": True},    {"label": "caching strategy",       "text": "cache invalidation strategy for a read-heavy service",         "inDomain": True},    {"label": "conflict with a peer",   "text": "handling disagreement with a teammate on a design decision",   "inDomain": True},    {"label": "system design scaling",  "text": "scaling a write-heavy service horizontally",                   "inDomain": True},    {"label": "sourdough starter",      "text": "keeping a sourdough starter alive and knowing when to feed it", "inDomain": False},    {"label": "css flexbox",            "text": "centring a div with CSS flexbox and why justify-content moves it", "inDomain": False},]if not (EMBED_UP and NODE_BIN):    print("DID NOT RUN — " + ("node is not installed on this image." if not NODE_BIN                              else "the embedding server is not up."))    log("demo_memory", "skipped", "no embedding server or no node")else:    out = run_bridge("memory", {"query": QUERY, "topics": TOPICS}, timeout=180)    if not out.get("ok"):        print("EMBEDDING FAILED — no similarity table is being shown.")        print(out.get("error"))        log("demo_memory", "failed", out.get("error", "")[:200])    else:        rows = sorted(out["rows"], key=lambda r: r["similarity"], reverse=True)        print(f"query          : {QUERY!r}")        print(f"vector dims    : {out['dimensions']}  (one vector per sentence — see --pooling mean)")        print(f"threshold      : {out['threshold']}  (src/memory.js DEFAULT_SIMILARITY_THRESHOLD)")        print(f"embed latency  : {out['latencyMs']/1000:.2f}s for {len(TOPICS)+1} vectors\n")        print(f"{'topic':<22}{'domain':<12}{'cosine':>9}   {'verdict':<10}")        print("-" * 60)        for r in rows:            verdict = "RETRIEVED" if r["similarity"] >= out["threshold"] else "filtered"            print(f"{r['label']:<22}{'in' if r['inDomain'] else 'OUT-OF':<12}{r['similarity']:>9.4f}   {verdict:<10}")        in_dom = [r["similarity"] for r in rows if r["inDomain"]]        out_dom = [r["similarity"] for r in rows if not r["inDomain"]]        print("-" * 60)        print(f"weakest in-domain    : {min(in_dom):.4f}")        print(f"strongest out-of-domain: {max(out_dom):.4f}")        print(f"gap                  : {min(in_dom) - max(out_dom):+.4f}")        log("demo_memory", "ran", f"gap={min(in_dom)-max(out_dom):.4f}")

**Being honest about that gap.** The separation between the weakest in-domain topic and the strongest out-of-domain one is real, but it is small — a 384-dimension embedding of a short phrase is not a precision instrument, and 'centring a div' shares more vocabulary with software interview topics than the label suggests. That is exactly why retrieval in the app is not the threshold alone: `retrieve()` also filters by `roleId`, so a backend session never sees a frontend session's records regardless of what the cosine says. A threshold this close to the noise floor is not something to lean on unaided.

## 11 · What we tried and disprovedEvery serious bug in this project produced confident, well-formed, completely wrong output.None of them looked like failures. The measurements below are from our own testing on theRTX 4050 dev box — they are quoted here as history, not produced by this notebook.| What we tried | What actually happened | Outcome ||---|---|---|| **Grading delivery from tone of voice** — the original pitch was that Gemma could hear hesitation vs confidence | Same words, spoken flat and spoken energetically, scored **90 and 90**. Scores swung **18 points between runs on the identical clip**. An earlier promising result turned out to be the model reading *filler words in the transcript*, not perceiving tone. | **Cut.** Delivery is now surfaced as descriptions — filler words per minute, pace, pauses — computed from timestamps, never asked of the model, with a visible off switch. Grading delivery would penalise nervous and non-native English speakers, who are exactly who this is for. || **GBNF grammar on open-ended question generation** | The model collapsed strings to `"..."` in roughly **1 run in 3**. `char*` gives it a cheap path to a valid-but-empty answer and it takes it. | **Scoped.** Grammars constrain extraction and grading, where the content is bounded. Generation is validated after the fact instead. || **Default sampling temperature on the grader** | The same answer scored **65, 75, 75** across three identical runs. | **Fixed.** `temperature: 0.0` in `evaluate()`. Re-ran: **65, 65, 65**. Cell 8 above re-proves this live. || **A grading regression test that built its own prompt** | It never imported the app's code. Every "regression" it reported across **two full rounds of debugging** was drift between two hand-maintained copies of the same prompt. The baseline score we had been defending had never measured anything. | **Fixed.** One module, imported by both production and tests. It is why this notebook shells out to `src/llm.js` instead of restating the prompt. || **The embedding server without `--pooling mean`** | It returned one vector *per token*; the parser took index `[0]`, which is the `[CLS]` token. Cosine similarity was comparing the first token of each string and returning **plausible numbers the entire time**. | **Fixed.** `--pooling mean` is mandatory and commented as such in cell 4. || **Memory that stored nothing** | The grading pipeline built its output objects without carrying `topic` through, so **every** record hit the empty-topic gate and was silently rejected. Unit tests passed — they used hand-built objects that included `topic`. | **Fixed.** Found only by tracing the real call chain by hand. |The through-line: none of these were crashes. Every one of them returned a well-formed,confident, wrong answer, and four of the six were invisible to a passing test suite.

## 12 · Offline proofThe project's second rule is that the app must work with the network physically switched off. This notebook is run with Kaggle's **Internet: Off** setting, and both model servers bind loopback. Below: the outbound attempt, and the actual listen sockets.

In [ ]:
import socketprint("--- outbound request attempt ------------------------------------------------")try:    with urllib.request.urlopen("https://example.com", timeout=10) as r:        print(f"UNEXPECTED: reached example.com with HTTP {r.status}.")        print("Internet is NOT off for this session. Turn it off in the notebook settings")        print("pane (Settings -> Internet -> Off) and re-run: the offline claim is the whole")        print("point of this project and it must be demonstrated, not asserted.")        OFFLINE = Falseexcept Exception as e:    print(f"https://example.com  ->  BLOCKED: {type(e).__name__}: {e}")    OFFLINE = Truetry:    socket.create_connection(("1.1.1.1", 443), timeout=5).close()    print("UNEXPECTED: raw TCP to 1.1.1.1:443 succeeded.")    OFFLINE = Falseexcept Exception as e:    print(f"tcp 1.1.1.1:443      ->  BLOCKED: {type(e).__name__}: {e}")print("\n--- listening sockets in this container --------------------------------------")rc, listeners = sh("ss -ltnp 2>/dev/null || netstat -ltnp 2>/dev/null")print(listeners.strip() or "(ss/netstat unavailable in this image)")print("\nBoth llama-server processes were launched with --host 127.0.0.1 (see cell 4).")print("Neither is reachable from outside this container, and nothing in this notebook")print("opens an outbound connection: every request above targets 127.0.0.1.")log("offline_proof", "ran", f"outbound_blocked={OFFLINE}")

## 13 · What this notebook is not- **There is no microphone in it, and there never will be.** No mic widget, no upload widget,  no file picker. It reads the fixture WAVs shipped in `mockmate-fixtures` and nothing else.  The capture path lives in `sidecar/`, on the user's own machine, and is not started here.  Nothing in this notebook uploads audio, transcripts, or scores anywhere.- **The latency numbers above are a Kaggle T4.** The project's target is a 6GB RTX 4050 and the  figures in the writeup were measured there. They are different machines; the numbers will not  match, and treating one as a substitute for the other would be a fabricated comparison. If the  build cell fell back to CPU, every latency figure above carries a CPU-only label and is not  comparable to either.- **The live product runs on the user's own machine.** This notebook exists so the project can  be evaluated by someone who does not have a GPU handy — it is a demonstration harness, not  the product. The product is a static web app plus two local servers, with no build step and  no dependencies.- **Run it yourself:** the Quickstart in  [the repo README](https://github.com/varadshajith/mockmate-gemma#-quickstart) is four commands —  build llama.cpp, fetch the GGUFs, start two `llama-server` processes, `python3 -m http.server`.  Then switch the Wi-Fi off and keep going.

In [ ]:
# --- run checklist ----------------------------------------------------------# Printed from RUN_LOG, which each cell wrote as it went. This is the only# honest answer to "what can we claim from this run?"print("=" * 78)print("RUN CHECKLIST — what executed live in this session")print("=" * 78)ORDER = [    ("env_probe",         "cell 2  environment probe"),    ("verify_inputs",     "cell 3  verify attached datasets"),    ("build_llamacpp",    "cell 4  build llama.cpp"),    ("launch_servers",    "cell 5  launch both servers"),    ("demo_transcribe",   "cell 7  transcription on fixture audio"),    ("demo_grading",      "cell 8  hybrid grading (audio + text)"),    ("demo_determinism",  "cell 9  determinism proof"),    ("demo_thresholds",   "cell 10 thresholds in JS"),    ("demo_memory",       "cell 11 memory retrieval"),    ("offline_proof",     "cell 13 offline proof"),]MARK = {"ran": "LIVE   ", "skipped": "SKIPPED", "failed": "FAILED "}for key, label in ORDER:    entry = RUN_LOG.get(key)    if entry is None:        print(f"  NOT RUN  {label:<44} (cell was not executed)")    else:        print(f"  {MARK.get(entry['status'], '?      ')}  {label:<44} {entry['detail']}")print("-" * 78)print(f"build mode         : {BUILD_MODE.upper() if BUILD_MODE else 'n/a'}"      f"{'  — latency numbers above are NOT comparable to GPU figures' if BUILD_MODE != 'cuda' else ''}")print(f"flash-attn         : {FLASH_ATTN}")live = [k for k, e in RUN_LOG.items() if e["status"] == "ran"]print(f"cells run live     : {len(live)} of {len(ORDER)}")not_live = [label for key, label in ORDER if RUN_LOG.get(key, {}).get("status") != "ran"]if not_live:    print("NOT demonstrated in this run — do not claim these from this notebook:")    for label in not_live:        print(f"  - {label}")else:    print("Every demo cell ran live in this session.")print("=" * 78)